# 02 – Data Cleaning

This notebook cleans the Craigslist used cars dataset (Kaggle, `austinreese/craigslist-carstrucks-data`) before exploratory analysis and modelling. The raw file has 295,451 listings and 41 columns. The goal is a clean table for predicting used car prices.

### Step 1 – Selecting columns
`size` and `county` are 100% empty, and `other_attributes` and `monthly_payment` are about 98% empty. Id, URL and image columns carry no information about price. I kept 25 relevant columns, which had 525,698 missing cells before cleaning.

In [1]:
import pandas as pd
import numpy as np

KEEP = ["price", "year", "manufacturer", "model", "condition", "cylinders", "fuel",
        "odometer", "title_status", "transmission", "VIN", "drive", "type",
        "paint_color", "description", "state", "region", "lat", "long",
        "posting_date", "seller_type", "image_count", "delivery_available",
        "odometer_broken", "odometer_rolled_over"]

df = pd.read_csv("../data/raw/vehicles.csv", usecols=KEEP, low_memory=False)
rows_before = len(df)
print("Shape before cleaning:", df.shape)
print("Missing cells before:", int(df.isna().sum().sum()))

Shape before cleaning: (295451, 25)
Missing cells before: 525698


### Step 2 – Fixing types and checking ranges
`price` was stored as text and ranges from 0 to 1e24 (median 12,500). `year` goes from 1900 to 2027 and `odometer` up to 10,000,000. I converted price to numeric and dates to datetime, and used the percentiles to set limits in Step 6.

In [2]:
df["price"] = pd.to_numeric(
    df["price"].astype(str).str.replace(r"[^0-9.]", "", regex=True), errors="coerce"
)
df["posting_date"] = pd.to_datetime(df["posting_date"], errors="coerce", utc=True)
print(df[["price", "year", "odometer"]].describe(percentiles=[.01, .05, .5, .95, .99]).round(0))

              price      year    odometer
count  2.954510e+05  295425.0    291415.0
mean   3.384656e+18    2013.0    114562.0
std    1.839743e+21      12.0    171310.0
min    0.000000e+00    1900.0         1.0
1%     0.000000e+00    1962.0        35.0
5%     9.950000e+02    1995.0     12000.0
50%    1.250000e+04    2015.0    107500.0
95%    4.999300e+04    2024.0    216141.0
99%    8.990000e+04    2026.0    295698.0
max    1.000000e+24    2027.0  10000000.0


### Step 3 – Checking categories
`condition` is 45.9% missing, `cylinders` 27.1%, `paint_color` 25.3% and `drive` 23.8%. Labels are inconsistent (e.g. "SUV" in capitals, "pickup" vs "truck"). I decided to standardise the text and keep missing categories as "unknown" instead of guessing.

In [3]:
for col in ["manufacturer", "condition", "cylinders", "fuel", "title_status",
            "transmission", "drive", "type", "paint_color", "seller_type"]:
    print(col, df[col].nunique())
    print(df[col].value_counts(dropna=False).head(15), "\n")

manufacturer 42
manufacturer
ford             51374
chevrolet        34006
toyota           27042
honda            17404
nissan           14375
ram              14297
jeep             13510
subaru           10840
NaN              10792
gmc              10678
mercedes-benz     9592
bmw               9572
dodge             7414
hyundai           6593
kia               6122
Name: count, dtype: int64 

condition 6
condition
NaN          135490
excellent     77943
good          51632
like new      19514
fair           6451
new            2426
salvage        1995
Name: count, dtype: int64 

cylinders 8
cylinders
NaN             80167
4 cylinders     77724
6 cylinders     72895
8 cylinders     54222
other            5937
3 cylinders      1570
5 cylinders      1410
10 cylinders     1236
12 cylinders      290
Name: count, dtype: int64 

fuel 5
fuel
gas         249863
diesel       25093
other         7642
hybrid        7454
electric      4268
NaN           1131
Name: count, dtype: int64 

title_

### Step 4 – Standardising text
I lowercased and stripped the text columns so the same category isn't counted twice. `cylinders` was text like "6 cylinders", so I extracted the number. Its remaining gaps will be imputed after the train/test split to avoid leakage.

In [4]:
text_cols = ["manufacturer", "model", "condition", "fuel", "title_status",
             "transmission", "drive", "type", "paint_color", "seller_type", "state", "region"]
for c in text_cols:
    df[c] = df[c].astype("string").str.strip().str.lower()

# cylinders looks like "6 cylinders", so extract the number
df["cylinders"] = pd.to_numeric(
    df["cylinders"].astype("string").str.extract(r"(\d+)")[0], errors="coerce"
)

### Step 5 – Duplicates
There were 0 exact duplicates, but matching on price, year, model, odometer, VIN, state and description found 47,299 reposted listings. I removed them so the model doesn't see the same car twice or leak it into the test set.

In [5]:
dup_cols = ["price", "year", "manufacturer", "model", "odometer", "VIN", "state", "description"]
print("Near-duplicates:", df.duplicated(subset=dup_cols).sum())
df = df.drop_duplicates(subset=dup_cols)

Near-duplicates: 47299


### Step 6 – Outliers and missing values
10,288 listings had price 0. Using the percentiles (price 99% = 89,900, odometer 99% = 295,698), I kept price 500–100,000, year 1990–2026 and odometer 1–500,000. I dropped rows missing price, year, manufacturer, model or odometer, and filled missing categories with "unknown". Filling odometer with the median was an alternative, but it would weaken its link to price.

In [6]:
df = df[df["price"].between(500, 100_000)]
df = df[df["year"].between(1990, 2026)]
df = df[df["odometer"].between(1, 500_000)]   # drops missing odometer too
df = df.dropna(subset=["price", "year", "manufacturer", "model"])

# Categorical gaps become "unknown" (no leakage risk)
for c in ["condition", "fuel", "title_status", "transmission", "drive", "type", "paint_color"]:
    df[c] = df[c].fillna("unknown")

### Step 7 – After cleaning
Rows went from 295,451 to 214,253 (81,198 removed) and missing cells from 525,698 to 144,911. The dataset still meets the 200,000-row requirement. The cleaned file is saved in `data/processed/` and isn't committed because of its size.

In [7]:
print("Rows before:", rows_before, "| after:", len(df))
print("Missing cells after:", int(df.isna().sum().sum()))
print(df.dtypes)

import os
os.makedirs("../data/processed", exist_ok=True)
df.to_parquet("../data/processed/vehicles_clean.parquet", index=False)

Rows before: 295451 | after: 214253
Missing cells after: 144911
region                               string
price                               float64
year                                float64
manufacturer                         string
model                                string
condition                            string
cylinders                             Int64
fuel                                 string
odometer                            float64
title_status                         string
transmission                         string
VIN                                     str
drive                                string
type                                 string
paint_color                          string
description                             str
state                                string
lat                                 float64
long                                float64
posting_date            datetime64[us, UTC]
seller_type                          string
image_count 

## Summary
The main problems were text prices, impossible values and 47,299 reposted listings. After cleaning, 214,253 clean rows remain. One limitation is that Craigslist prices are asking prices, not final sale prices.